# Regex
> Search and replace text with compiled regular expressions

> Adapted from the upstream BasedPL docs notebook. The `]help` magic cell is omitted (REPL-only, not executable BPL). `•load` works against the bundled `regex.bpl` in this folder.

Natively, patterns use [Rust regex syntax](https://docs.rs/regex/latest/regex/#syntax). In the browser they use [JavaScript's](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Regular_expressions). `p.matches` gives a table of the matches, as a record of columns. Here `p` finds runs of digits:

In [ ]:
p←•r "[0-9]+"
p.matches "abc123def45"

["text":["123" "45"] "position":[3 9]ₓ "groups":[0⍴⊂"" 0⍴⊂""]]

Matches don't overlap. Positions count characters, and `≢¨` of the `text` column gives each match's length. A text with no match gives a table with no rows:

In [ ]:
m←(•r "é|🐈+").matches "aé🐈🐈z"
m.position
≢¨m.text
(•r "é|🐈+").matches "abc"

[1 2]ₓ

[1 2]ₓ

["text":0⍴⊂"" "position":⍬ₓ "groups":0⍴⊂0⍴⊂""]

The `groups` column holds the captured strings of each match, without the whole match. An unmatched optional capture gives `""`:

In [ ]:
p←•r "([a-z]+)([0-9]+)"
(p.matches "ab12 cd3").groups

("ab" "12" ⋄ "cd" "3")

`template p.replace text` replaces each match with `template`. In a native template, `$0` is the whole match and `$$` is a literal `$`. An unknown or unmatched capture expands to `""`. In the browser, templates use JavaScript's syntax, where `$<word>` names a group:

In [ ]:
p←•r "(?<word>[a-z]+)([0-9]+)"
"${word}:$2:$$" p.replace "ab12"

ab:12:$

Natively, inline flags change how a pattern matches -- `(?i)` ignores case, for example. **In the browser, patterns compile with JavaScript's native `RegExp`, which has no inline-flag syntax**, so this example uses a character class instead, to the same effect. Use Each to search several strings:

In [ ]:
p←•r "[Cc][Aa][Tt]"
{≢(p.matches ⍵).text}¨"Cat cat" "dog"

[2 0]ₓ

Empty matches occur at character boundaries:

In [ ]:
p←•r ""
(p.matches "é🐈").position
'-' p.replace "é🐈"

[0 1 2]ₓ

-é-🐈-

`lib/regex.bpl` supplies `pattern f regex_replace text`. It applies `f` to each matched string and keeps the text between matches:

In [ ]:
[regex_replace]←•load "examples/regex.bpl"
"[0-9]+" {⍕2×•json ⍵}regex_replace "a12 b3"

a24 b6